# Majorant\_v2 — open cascades on the conservative engine

This notebook runs the two open configurations, coagulation and fragmentation, on
`BF_v_no_resampling_v2.py`. One simulated particle stands for exactly one physical
particle for the whole run, so the total number and the total mass are conserved to
machine precision and the reported `mass_drift` is identically zero; anything else is
a bug rather than a fluctuation. That is what *conservative* means in the names of
the files this notebook writes.

An open system has a source and an absorbing boundary, so a genuine steady state
exists and the spectrum to measure is the **instantaneous** one rather than a
superposition over time. $\Delta t$ never enters the estimator, which is why the
closed notebook has to agonise over the snapshot cadence and this one does not — and
which is also why every snapshot taken after the cascade reached the sink is an
independent draw from the same distribution, so they are averaged rather than thrown
away. That average costs nothing and is worth a factor $\sqrt{K}$ in noise. The theory
under test is

$$\alpha = -\frac{3+\lambda}{2}, \qquad b = \frac{2}{1-\lambda},$$

with $\lambda$ the homogeneity degree of the kernel, $\alpha$ the slope of $dN/dm$
across the inertial range and $b$ the growth exponent read from the isochrones.
Coagulation injects monomers at $m_{\rm inj}$ and absorbs products above
$m_{\rm sink}$; fragmentation injects large bodies at $m_{\rm inj}$ and absorbs
fragments below $m_{\rm sink}$. The two share $\alpha$ and $b$ exactly, and only the
direction of the drift changes.

Spectra are drawn compensated, as $m^2\,dN/dm$ — the mass per logarithmic mass
interval, whose slope is $\alpha+2$. Every fit is performed on the raw $dN/dm$ and
the compensation is applied at draw time only, so the reported index is never the
compensated one.

Each successful run is written into `runs/` by `add_last_run`, keeping the spectra,
the isochrone histogram and all the parameters but discarding the per-particle
arrays. The analysis notebooks in that folder rebuild every figure from those files,
so a plot can be changed without paying for the simulation again.


## How the index is measured

A measured spectrum is a power law only *between* the two characteristic masses of
the problem. Outside that band it bends — at the injection scale because of the
source, at the sink because of truncation and vanishing statistics — and a single fit
across the whole array averages the plateau together with both bends, returning a
number that describes neither. What is measured here instead is the plateau in the
local slope,

$$\Gamma(m)=\frac{d\log F}{d\log m},$$

computed by least squares on a sliding window. `BF.find_inertial_range` returns the
longest contiguous stretch over which $\Gamma$ stays flat within a tolerance,
together with that stretch's mean, its scatter and its width in decades. The mean is
the measured $\alpha$: it is what enters the summary table and it is the line drawn
over the spectrum. When it comes back as `nan` the run simply has no inertial range
yet, which is the correct answer rather than a failure. What it is computed on is
`steady_spectrum`, the mean of the snapshots taken after the sink gate opened, not a
single snapshot.

Running alongside it is the a-priori guard band, half a decade stripped from each end
of the interval between $m_{\rm inj}$ and $m_{\rm sink}$, chosen before anyone looks
at the data. The two must agree; where they do not, the cascade has not developed
over the range that was assumed. The width in decades is printed next to every index
because a plateau narrower than roughly one decade is not a power law however tight
its error bar looks, and because the plateau finder can only test whether the curve
you computed has a scaling region — not whether that curve is the right quantity to
have computed.


In [ ]:
import os, importlib, inspect, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import BF_v_no_resampling_v2 as BF          # <-- v2 baseline engine, w == 1
importlib.reload(BF)

plt.rcParams.update({"figure.dpi":110, "font.size":9, "axes.grid":True,
                     "grid.alpha":0.25, "figure.figsize":(9,3.2)})

KERNEL = BF.kernel_geometric         # lambda = 2/3 (geometric cross section)
LAM    = BF.KERNEL_LAMBDA[KERNEL.__name__]
EDGES  = 10.0 ** np.arange(-4, 12, 0.1)   # top edge 10^7.9; see check_grid below


assert "stop_sink_events" in inspect.signature(BF.simulate).parameters, \
    "the engine on the path is not BF_v_no_resampling_v2.py"

print("engine =", BF.__name__)
print("kernel =", KERNEL.__name__, " lambda =", LAM)
PR = BF.predict("open", LAM)
print("  open   beta = %.4g   b = %.4g   alpha = %.4g" % (PR["beta"], PR["b"], PR["alpha"]))
print("  grid   %.3g .. %.3g  (%d bins of %.2f dex)"
      % (EDGES[0], EDGES[-1], EDGES.size-1, np.log10(EDGES[1]/EDGES[0])))

# Age bins must be chosen from b, not copied from the lambda = 0 notebook.  One age
# bin of width D_tau dex covers b * D_tau dex of MASS, so at b = 6 the old 0.15 dex
# step jumps 0.9 dex of mass per bin and the growth-law mask catches two bins.
# Fix the MASS resolution instead and let the age step follow.
AGE_STEP = 0.30 / PR["b"]          # 0.15 at b=2 (identical to the constant-kernel run)
print("  ages   %.3f dex per bin  ->  %.2f dex of mass per bin" % (AGE_STEP, 0.30))

RESULTS = {}   # collected for the summary table at the end


In [ ]:
# ----------------------------------------------------------------------
#  The measurement layer
# ----------------------------------------------------------------------
#  Everything that turns arrays into a number now lives in BF_analysis.py, next to
#  this notebook: the estimator and its error bars, the plateau finders, the
#  isochrone machinery and the saver.  Keeping it there rather than in a cell means
#  the run notebooks and the six analysis notebooks cannot drift apart, and that a
#  change to the estimator is one edit rather than nine.
#
#  The names are aliased below so the figure cells further down read exactly as
#  before.

import BF_analysis as AN

anchor_amplitude = AN.anchor_amplitude
pick_isochrones  = AN.pick_isochrones
draw_isochrones  = AN.draw_isochrones
compensated_ylim = AN.compensated_ylim
add_last_run     = AN.add_last_run


def steady_spectrum(run, frac=0.5):
    """
    (F, K): the averaged post-gate spectrum and how many snapshots went into it.

    AN.steady_spectrum returns the full dict -- F, the empirical and Poisson error
    bars, the raw counts and K_eff.  This shim keeps the two-value call used below;
    call AN.steady_spectrum directly when the error bars are wanted.
    """
    s = AN.steady_spectrum(run, frac=frac)
    return s["F"], s["K"]


def check_grid(*masses):
    """Refuse to start if a characteristic mass falls outside EDGES."""
    return AN.check_grid(EDGES, *masses)


---
## Open system with coagulation

Monomers are injected at $m_{\rm inj}$ at a constant rate and products crossing
$m_{\rm sink}$ are removed. The injection rate follows from the balance of particle
number in the steady state, where the accepted event rate must equal the injection
rate:

$$q = \frac{\langle K\rangle\,N_{\rm ss}^{2}}{2V}.$$

Writing $q=N^2/2$, as the original four-case notebook did, is that same expression at
$\langle K\rangle = 1$ — true for the constant kernel and for nothing else. Because
the population is dominated by the injection scale whenever $\alpha<-1$, most pairs
are $(m_{\rm inj},m_{\rm inj})$ and $\langle K\rangle$ is of order
$K(m_{\rm inj},m_{\rm inj})$, with heavier partners raising it by roughly a factor
two. `live` therefore settles somewhat below the nominal target, and that is fine;
what matters is that it settles at all and that $M_{\rm out}/M_{\rm in}$ heads
towards one.

Nothing in an open run may be gated on an absolute time. The clock advances by
$\Delta t = 2V/(wR)$ with $R\propto N^2$, so after $E$ trials

$$t \simeq \frac{2E}{N^{2}},$$

and raising $N$ at a fixed event budget makes the run *shorter* in physical time
rather than longer. A gate expressed as a time and tuned at one $N$ is never crossed
at ten times that $N$. Both gates used below are therefore counted in sink
absorptions: the isochrones begin accumulating once `iso_start_sink` particles have
crossed the entire inertial range, and the run ends after `stop_sink_events` of them.
Each is a statement about the physics and needs no retuning when $N$, the mass range
or the kernel changes. A ceiling on `max_events` is kept alongside them, not as the
criterion but as the guarantee that a mistuned $q$ cannot cost an afternoon;
`stop_reason` afterwards says which of the two actually bit.

The isochrone age bins are derived from $b$ rather than copied from a run at another
kernel. One age bin of width $\Delta_\tau$ dex spans $b\,\Delta_\tau$ dex of mass, so
the mass resolution is fixed at $0.3$ dex and the age step follows as
$\Delta_\tau = 0.3/b$.


In [ ]:
# OPEN system: the steady state is the LAST snapshot, dt never enters the estimator,
# so snapshotting by events is perfectly fine here.
#
# The isochrone gate is PHYSICAL, not temporal: bin ages only after the sink has swallowed
# N_OUT particles.  In v2 that gate is armed by default (10) for BOTH processes; it is passed
# explicitly below only to keep the number visible next to the budget it implies.

N_SS   = 1e6
M_INJ  = 1.0
M_SINK = 1e6
N_OUT  = 10                                               # sink absorptions before binning ages
N_STOP = 100                                              # sink absorptions before STOPPING (see below)
MAX_EV = int(3 * (N_SS*np.sqrt(M_SINK) + N_STOP*M_SINK))  # resource ceiling, whichever comes first
check_grid(M_INJ, M_SINK)

# --- injection rate: the one constant that does NOT survive a change of kernel ------
# Number balance in steady state:  q = accepted event rate = <K> N^2 / (2V).
# The original notebook wrote q = N^2/2, which is that formula with <K> = 1 -- true for
# the CONSTANT kernel and for nothing else.  N is dominated by the injection scale
# (alpha < -1), so most pairs are (m_inj, m_inj) and <K> is of order K(m_inj, m_inj);
# heavier partners push it up by roughly a factor two, so `live` settles somewhat below
# N_SS.  That is fine.  What matters is that panel (a) shows a PLATEAU and M_out/M_in
# heads for 1 -- retune K_TYP by hand if it drifts.
K_TYP  = KERNEL(M_INJ, M_INJ)                  # 1.0 for the constant kernel, 4.0 here
q3     = 0.5 * K_TYP * N_SS**2
t_c    = 1.0 / (K_TYP * N_SS)                  # collision time per particle: 1/(K n), V = 1
print("K(m_inj,m_inj) = %.4g   ->   q = %.3e,   t_c = %.3e" % (K_TYP, q3, t_c))

# Cost, measured for THIS kernel: 100 absorptions at m_sink = 1e5 take ~1.2e7 events.
# The notebook's own estimate  E ~ N*sqrt(m_sink) + n_out*m_sink  was derived at
# lambda = 0 and overshoots here, because a geometric kernel drives mass to the sink
# faster per event.  Both brakes are armed and sized to bite at about the same place;
# r3["stop_reason"] then says which one actually did.

# Cadence is sized from the EXPECTED cost, not from the ceiling.  The sink term
# dominates and is the only one that has to be paid, so N_STOP*M_SINK is a
# deliberate LOWER bound: too many snapshots cost one histogram pass each, too
# few cost the measurement.  Sizing the stride off MAX_EV instead gave seven
# snapshots on a run that ended twenty-seven times before the ceiling.
E_EXP  = N_STOP * M_SINK
print("brakes: stop_sink_events = %d   |   max_events = %.2e" % (N_STOP, MAX_EV))
print("cadence: expected %.2e events -> stride %.2e, about 200 snapshots"
      % (E_EXP, max(int(E_EXP)//200, 1)))

r3 = BF.simulate(process="coagulation", system="open", kernel=KERNEL, edges=EDGES,
                 ic={"m":M_INJ, "N":N_SS}, injection_rate=q3, injection_mass=M_INJ,
                 sink_mass=M_SINK,
                 snapshot_mode="events", snapshot_stride=max(int(E_EXP)//200, 1),
                 # max_events=np.inf is only safe when SOMETHING ELSE can stop the run.
                 # In an OPEN COAGULATION run nothing else can: live sits on a plateau so
                 # `live < 2` never fires, stop_max/min_mass are None, and the stall guard
                 # stays quiet because events keep being accepted.  Without the line below
                 # this cell runs forever.  stop_sink_events is the v2 brake and, unlike
                 # max_events, it means the same thing at every kernel -- but keep a
                 # resource ceiling as well, so a mistuned q cannot cost an afternoon.
                 max_events=MAX_EV, stop_sink_events=N_STOP,
                 iso_age_edges=10.0**np.arange(-3, 3, AGE_STEP) * t_c,
                 iso_start_sink=N_OUT,          # the gate (v2 default is 10 anyway)
                 age_rule="mass_weighted",      # report this; try 'min' to test sensitivity
                 rng=np.random.default_rng(3), verbose=True)

print("live %d -> %d   (must plateau)" % (r3["live"][0], r3["live"][-1]))
print("sink absorptions = %d  (gate asks for %d) | M_out/M_in = %.3f  (must approach 1)"
      % (r3["sink_events"], N_OUT, r3["M_out"][-1]/max(r3["M_in"][-1],1)))
print("isochrones: %d snapshots, accumulation began at t = %.3g  (t*N = %.0f)"
      % (r3["iso_snapshots"], r3["iso_t_begin"], r3["iso_t_begin"]*N_SS))
print("run ended at t = %.3g  (t*N = %.0f) | stop = %s"
      % (r3["t"][-1], r3["t"][-1]*N_SS, r3["stop_reason"]))

# ---------------- the spectrum -------------------------------------------------
F3, NSS3 = steady_spectrum(r3)
print("estimator: mean of %d snapshots taken after the sink gate" % NSS3)

# THE measurement: the longest plateau in the local slope.  Everything downstream --
# the model on panel (b), the summary row -- reads this and nothing else.
ir3 = BF.find_inertial_range(r3["centers"], F3)

# a-priori band, kept only as an independent cross-check.  NOTE the upper scale is M_SINK,
# per guard_band's own docstring (open system: [m_inj, m_sink]).
gb3 = BF.guard_band(M_INJ, M_SINK, 0.5)
f3  = BF.fit_powerlaw(r3["centers"], F3, *gb3)
pr3 = PR

# ---------------- the growth law, from the isochrones --------------------------
tau3, mbar3, n3 = BF.iso_mean_mass(r3["iso_counts"], r3["centers"], r3["iso_age_edges"])
# The window was hard-coded (3, 300) for the constant kernel.  Tie it to the guard band
# instead: fit the growth law only where the spectrum itself is a clean power law.  At
# b = 6 a 2-decade mass window is a THIRD of a decade in tau and catches two age bins.
k3 = np.isfinite(mbar3) & (n3 > 1e3) & (mbar3 > gb3[0]) & (mbar3 < gb3[1])
b3 = np.polyfit(np.log(tau3[k3]), np.log(mbar3[k3]), 1)[0] if k3.sum() > 3 else np.nan

print("\nspectrum, AUTO PLATEAU [%.3g,%.3g] : alpha = %+.3f +- %.3f  over %.2f decades   <== the answer"
      % (ir3["m_lo"], ir3["m_hi"], ir3["alpha"], ir3["scatter"], ir3["decades"]))
print("spectrum, guard band   [%.3g,%.3g] : alpha = %+.3f   R2 = %.3f   (cross-check)"
      % (gb3[0], gb3[1], f3["alpha"], f3["r2"]))
print("theory                              : alpha = %+.3f   b = %.3f" % (pr3["alpha"], pr3["b"]))
print("isochrone growth                    : b = %.3f   [%d age bins]" % (b3, k3.sum()))
if k3.sum() <= 3:
    print("  !! too few usable age bins: %d iso snapshots, %d sink absorptions."
          "  Raise N_STOP (cost ~ N*sqrt(m_sink) + n_out*m_sink) or lower M_SINK."
          % (r3["iso_snapshots"], r3["sink_events"]))

RESULTS["3 open coag"] = dict(b=b3, b_th=pr3["b"], alpha=ir3["alpha"], alpha_th=pr3["alpha"],
                              scatter=ir3["scatter"], decades=ir3["decades"],
                              alpha_gb=f3["alpha"])


# ---------------- persist, for the analysis notebooks in runs/ -----------------
KTAG      = KERNEL.__name__.replace("kernel_", "")
RUN_NAME3 = "open_coagulation_conservative_%s" % KTAG
add_last_run(r3, RUN_NAME3, analysis=dict(
    alpha_plateau=ir3["alpha"], alpha_scatter=ir3["scatter"],
    plateau_decades=ir3["decades"], plateau_m_lo=ir3["m_lo"], plateau_m_hi=ir3["m_hi"],
    alpha_guard=f3["alpha"], guard_r2=f3["r2"], guard_lo=gb3[0], guard_hi=gb3[1],
    alpha_theory=pr3["alpha"], b_theory=pr3["b"], b_iso=b3,
    iso_age_bins_used=k3.sum(), lam=LAM, N_ss=N_SS, m_inj=M_INJ, m_sink=M_SINK,
    q=q3, K_typ=K_TYP, t_c=t_c, age_step=AGE_STEP,
    n_out_gate=N_OUT, n_out_stop=N_STOP))


In [ ]:
N_ISO_SHOW = 1000      # <<-- how many isochrones to draw on panel (b).  The line below prints
                    #      how many are available first, so you always know what you are seeing.
                    #      NOTE: AGE_STEP is now 0.05 dex (b = 6), so there are ~119 age bins
                    #      instead of 39 and "100" really does draw ~70 curves.  Drop to ~12
                    #      if panel (b) turns into a hairball.

idx3, tau3c, navail3 = pick_isochrones(r3, N_ISO_SHOW)
print("isochrones: %d of %d age bins carry statistics -> drawing %d"
      % (navail3, len(tau3c), len(idx3)))

fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.2))

# ---- (a) steady-state check, with the sink counter on the right axis ----------
ax[0].plot(r3["t"], r3["live"], ".-", ms=3, lw=.8, color="C0")
ax[0].set_xlabel("t"); ax[0].set_ylabel("live particles", color="C0")
ax[0].set_ylim(0, 1.2*np.max(r3["live"]))      # start at zero: no offset trick on this axis
ax[0].set_title("(a) steady-state check")
axr = ax[0].twinx(); axr.grid(False)
axr.plot(r3["t"], r3["n_out"], "-", lw=1.3, color="C1")
axr.set_ylabel(r"absorbed at sink  $n_{\rm out}$", color="C1")
if np.isfinite(r3["iso_t_begin"]):
    ax[0].axvline(r3["iso_t_begin"], ls="--", lw=1, color="C3")
    ax[0].text(r3["iso_t_begin"], ax[0].get_ylim()[1], " isochrones start\n (%d sink hits)" % N_OUT,
               fontsize=6, va="top", color="C3")

# ---- (b) COMPENSATED spectrum: m^2 dN/dm = mass per logarithmic mass interval --
c = r3["centers"]
a = ax[1]
draw_isochrones(a, r3, idx3, tau3c, tau_scale=N_SS)
if len(idx3):
    # completeness check: the isochrones must add up to the steady state
    iso_sum = r3["iso_dndm"].sum(axis=0) / max(r3["iso_snapshots"], 1)
    a.loglog(c, np.where(iso_sum > 0, iso_sum*c**2, np.nan), "-", lw=3.5, alpha=.35,
             color="0.4", zorder=2, label="sum of isochrones")
a.loglog(c, np.where(F3 > 0, F3*c**2, np.nan), "o", ms=3, color="k", zorder=4,
         label="steady state")

xs  = np.logspace(np.log10(ir3["m_lo"]), np.log10(ir3["m_hi"]), 30)
A_m = anchor_amplitude(c, F3, ir3["m_lo"], ir3["m_hi"], ir3["alpha"])
A_t = anchor_amplitude(c, F3, ir3["m_lo"], ir3["m_hi"], pr3["alpha"])
a.loglog(xs, A_m * xs**ir3["alpha"] * xs**2, "-",  lw=2.2, color="C3", zorder=5,
         label=r"plateau $\alpha=%.2f$" % ir3["alpha"])
a.loglog(xs, A_t * xs**pr3["alpha"] * xs**2, "--", lw=1.4, color="C1", zorder=5,
         label=r"theory $%.2f$" % pr3["alpha"])
compensated_ylim(a, c, F3)
a.set_xlim(.5, 3e8); a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
a.legend(fontsize=6, loc="lower left"); a.set_title(r"(b) $m^2\,dN/dm$ + isochrones")

# ---- (c) isochrone growth law -------------------------------------------------
if k3.sum() > 3:
    ax[2].loglog(tau3[k3], mbar3[k3], "o", ms=3)
    ax[2].loglog(tau3[k3], mbar3[k3][0]*(tau3[k3]/tau3[k3][0])**b3, "-", lw=1,
                 label=r"$\langle m\rangle\propto\tau^{%.2f}$ (theory %.0f)" % (b3, pr3["b"]))
    ax[2].legend(fontsize=7)
else:
    ax[2].set_xscale("log"); ax[2].set_yscale("log")
    ax[2].text(.5, .5, "no usable isochrones\n%d iso snapshots, %d sink hits"
               % (r3["iso_snapshots"], r3["sink_events"]),
               ha="center", va="center", fontsize=7, color="C3", transform=ax[2].transAxes)
ax[2].set_xlabel(r"age $\tau$"); ax[2].set_ylabel(r"$\langle m\rangle$")
ax[2].set_title("(c) isochrone growth law")
fig.tight_layout()


---
## Open system with fragmentation

Large bodies are injected at $m_{\rm inj}$ and fragments falling below
$m_{\rm sink}$ are absorbed — the mirror of the coagulation case, predicting the same
$\alpha$. This one is run twice, because the rule deciding which particle breaks is a
choice of physics rather than a detail of implementation.

If the heavier of the pair always breaks, whatever hit it, then the disruption rate
of a body of mass $m_0$ against a background $F(m)\propto m^{\alpha}$ is

$$\nu(m_0)\sim\int_{m_{\rm sink}}^{m_0}F(m')\,K(m_0,m')\,dm'
        \sim\int_{m_{\rm sink}/m_0} x^{\alpha}\,dx ,$$

which diverges at the lower limit whenever $\alpha<-1$. The drift is then set by the
smallest particles in the box — by the sink scale rather than by $m_0$ — the
mean-field closure is void, and the measured index locks onto $-2$ for any kernel at
all. That is a boundary effect wearing the costume of universality.

The cure is the one Dohnanyi built in: a catastrophic-disruption threshold, on the
grounds that a grain of dust does not shatter a boulder. Requiring
$m_{\rm small}\ge f\,m_{\rm large}$ with $f$ of order a tenth to one restricts the
integral to $x\in[f,1]$, which is scale free, and recovers the predicted index. A
closed system is immune by construction, because its packet is narrow and every
partner is already of order $m_0$ — which is why the closed runs agree with the
theory at $f=0$ and the open ones do not.

The cell below runs the local rule only, $f=0.3$. The $f=0$ branch has been dropped:
without a threshold there is no steady cascade to measure at all — the last such run
ground its whole population into $1<m<447$ and returned $\alpha=-4.7$ over nine tenths
of a decade, which is the shape of a pile at the sink and not an index.


In [ ]:
# ----------------------------------------------------------------------------
#  Six decades: injected at m_inj = 1e6, absorbed at m_sink = 1.
# ----------------------------------------------------------------------------
#  Every fragmentation event creates exactly one particle, so the run obeys
#
#      live = N0 + n_injected + events - n_out
#
#  to the last unit.  Two consequences fix every number in this cell.
#
#  (1) One injected body becomes R = m_inj/m_sink = 1e6 fragments.  It therefore
#      costs 1e6 events to grind down and delivers 1e6 particles to the sink.  The
#      event counter and the sink counter are one budget seen from two ends: once the
#      cascade is running,  events = N_ss + n_out.  Asking for a number of particles
#      at the sink IS asking for a number of events, and the two brakes below are
#      set from each other rather than guessed.
#
#      WATCH THIS IDENTITY WHILE THE RUN PRINTS.  If `events` outruns `sink` by a
#      growing factor, `live` cannot plateau -- the population is being fed faster
#      than the sink drains it, and the run is a transient however long you wait.
#      Seen in practice at w = 0.05: events = 1.8e7 against sink = 3.5e6, live at
#      14e6 against a target of 4e6, <m> stuck near 13 while m_sink = 1.  With no
#      small chips there is no short path to the sink, so the whole population has
#      to be ground down together.  Either lower q4 or widen the split.
#
#  (2) The steady-state population N_ss, not the event budget, sets the memory.  Per
#      injected body the ledger reads +1 injected, +(R-1) splits, -R absorbed = 0, so
#      the particle array grows to N_ss and then stops however long the run goes.
#      N_ss = 4e6 is about 150 MB of arrays; that is the real ceiling on the range.
#
#  N_ss is therefore chosen FIRST and the injection rate follows from the number
#  balance q R = K(m_sink,m_sink) N_ss^2 / 2V -- the event rate evaluated where the
#  NUMBER lives, which for alpha < -1 is the sink scale, not the injection scale.
#
#  The previous version had this backwards.  It set the target population to N0*R,
#  with a comment sized for four decades while the range was six, so it asked for
#  4e8 particles and an injection rate a hundred times too large, and then bought
#  5e6 events where those parameters needed 1e10.  The run stopped on max_events at
#  M_out/M_in = 1e-7: no steady state, and the "break at 1e3" in the spectrum was not
#  a spectrum at all but the descending front, sitting at the mean mass
#  m_inj * n_injected / events.
#
#  THE AGE RULE SWEEP.  This cell now runs once per entry in AGE_RULES4, at the SAME
#  seed.  frag_age_rule decides which fragment keeps the parent's clock:
#
#      'inherit'  both keep it.  No clock is ever reset, so an "age class" is the
#                 whole descendant tree of one injected body -- a FAMILY, as wide as
#                 the inertial range by construction, however long the run.  This is
#                 what every earlier fragmentation run measured, and it is why the
#                 isochrones on those figures spanned six decades.
#      'heavier'  the heavier piece keeps it, the lighter is reborn at t_phys.  The
#                 tracer is then the always-heavy chain: a characteristic of the
#                 transport equation, i.e. the FRONT.  This is the one to measure.
#
#  The rule consumes no random numbers and touches no mass, so the two runs must
#  agree in live, sink_events and M_sys to the last unit.  That check is printed
#  below and is the whole reason for using one seed: if it fails, the patch has
#  leaked into the dynamics and nothing else in this notebook means anything.
import time

AGE_RULES4 = ["inherit", "heavier"]     # 'inherit'|'heavier'|'lighter'|'both_new'

M_INJ4, M_SINK4 = 1.0e6, 1.0
R4       = M_INJ4 / M_SINK4             # 1e6 fragments per injected body
N_SS4    = 4.0e6                        # <<-- target live population; this sets memory
N04      = 4                            # start AT the steady-state mass, not 100x above
W_SPLIT  = 0.5                         # <<-- полуширина дробления; 0.5 = как было

#  Fragment split: xi uniform on (0.5 - w, 0.5 + w).  w = 0.5 is the old uniform
#  split on (0,1) exactly.  w = 0.25 halves the isochrone width, from 0.58 to 0.27
#  decades, while keeping the generations overlapped -- at w = 0.05 they do not
#  overlap and the spectrum picks up a factor-2.3 picket fence every 0.3 dex.
#  It costs a LONGER TRANSIENT: with no small chips there is no shortcut to the
#  sink, so first breakthrough is late.  Measured on a probe: 300k events gave
#  1500 absorptions at w = 0.5 and none at all at w = 0.25.
#
#  NOTE, now that frag_age_rule exists: narrowing w was an attempt to sharpen the
#  isochrone through the physics of the split.  Under 'heavier' the tracer is narrow
#  by construction, so that job is done elsewhere and w can go back to 0.5.  The comb
#  is a SYSTEMATIC on alpha, not scatter -- a plateau fitted straight through it
#  returns a shifted index with a small error bar.

#  THE BRAKE IS THE SINK COUNTER, because that is the quantity with a meaning: one
#  injected body delivers R = 1e6 particles to it, so n_out/R is the number of bodies
#  the run has actually ground from end to end, and n_out/N_ss is how many times the
#  box has been replaced.  What each choice costs, at the ~2e4 events/s this engine
#  does (measured below on your own machine, so the printed numbers are yours):
#
#      n_out      bodies ground   turnovers   events    time per run
#      5.0e6            5             1.2      9e6        ~7 min
#      2.0e7           20             5        2.4e7      ~18 min
#      7.5e7           75            19        8e7        ~1 h
#
#  Below roughly ten turnovers the isochrones stay degenerate.  The box still
#  remembers t = 0, every particle carries the same age, and an "isochrone" is then a
#  snapshot of the spectrum rather than an age class -- which is exactly what the
#  previous run was showing.  That, not the spectrum, is what the two hours buy.
#
#  Multiply the last column by len(AGE_RULES4): the sweep runs the whole thing once
#  per rule, and HOURS4 below is the ceiling PER RULE, not for the cell.
F_RATIO4 = 0.3                          # <<-- disruption threshold; NON-LOCAL f = 0 is gone
N_OUT4   = 7.5e7                        # <<-- particles absorbed at the sink
HOURS4   = 2.0                          # <<-- wall-clock ceiling for the run, per rule
SEED4    = 4                            # identical for every rule -- see the check below
check_grid(M_INJ4, M_SINK4)

q4      = 0.5 * KERNEL(M_SINK4, M_SINK4) * N_SS4**2 / R4
t_turn4 = N_SS4 / (q4 * R4)             # box turnover: N_ss / (event rate).  The age
t_c4    = t_turn4                       # scale the isochrones are binned on.

#  Age grid.  It reaches five decades BELOW t_turn, not two: once clocks are reset
#  mid-cascade the age of a reborn fragment is a single collision time, orders of
#  magnitude under the turnover.  On the old (-2, 4) grid 'heavier' would pile every
#  tracer into the first bin and look like a failure of the rule rather than of the
#  binning.  After this run, look at which bins are occupied and re-centre.
AGE_EDGES4 = 10.0**np.arange(-5, 4, AGE_STEP) * t_turn4

# Calibrate on THIS machine rather than assuming a rate.  Two hours is a wall-clock
# request, and events per second is a property of the laptop, not of the physics.
# The rate is flat in `live` (measured: from 1e5 to 1e6 particles it does not move), so
# a short probe is enough -- but it MUST be run at the same f and w as the real thing.
# The disruption threshold rejects collisions after the majorant test, so it shows up
# as a lower acceptance and a lower event rate: measured 13000 ev/s at f = 0, w = 0.5
# against 9443 ev/s at f = 0.3, w = 0.25, i.e. the wrong probe is 38% optimistic.
#
# CAVEAT the probe cannot see: it starts from N04 = 4 bodies, where `live` is tiny and
# acceptance is high.  If the real run's population overshoots N_ss the rate falls and
# the ceiling below is optimistic.  Compare RATE4 against the cpu column while the run
# prints; a factor of three means the calibration was measured in a different regime.
_t0  = time.time()
_cal = BF.simulate(process="fragmentation", system="open", kernel=KERNEL, edges=EDGES,
                   ic={"m": M_INJ4, "N": N04},
                   frag_min_ratio=F_RATIO4, frag_split_width=W_SPLIT,
                   injection_rate=q4,
                   injection_mass=M_INJ4, sink_mass=M_SINK4, snapshot_mode="events",
                   snapshot_stride=50_000, max_events=200_000,
                   rng=np.random.default_rng(0), verbose=False)
RATE4 = 2.0e5 / max(time.time() - _t0, 1e-9)
del _cal

CEIL4   = int(RATE4 * 3600.0 * HOURS4)              # wall-clock ceiling, per rule
MAX_EV4 = int(min(CEIL4, N_SS4 + N_OUT4))           # whichever comes first

print("machine   : %.3g events/s  ->  %.3g events, %.2f h per rule (%d rules)"
      % (RATE4, MAX_EV4, MAX_EV4 / RATE4 / 3600.0, len(AGE_RULES4)))
print("            %s bites first"
      % ("the sink brake" if N_SS4 + N_OUT4 <= CEIL4 else
         "THE CLOCK -- lower N_OUT4 or raise HOURS4"))
print("population: N_ss = %.3g  ->  q4 = %.4g,  turnover t = %.4g,  memory ~ %.0f MB"
      % (N_SS4, q4, t_turn4, N_SS4 * 24 * 1.6 / 1e6))
print("brake     : n_out = %.3g absorptions = %.1f bodies fully ground = %.1f turnovers"
      % (N_OUT4, N_OUT4 / R4, N_OUT4 / N_SS4))
print("            events = N_ss + n_out = %.3g  (this is the identity, not an estimate)"
      % (N_SS4 + N_OUT4))
print("acceptance: M_out/M_in -> 1, live flat near %.3g, <m> stops falling near %.0f"
      % (N_SS4, 50))

# One run per rule, one seed.  The f = 0 branch is gone: without a disruption
# threshold the grinding rate of a big body is set by the smallest particles in the
# box, the mean-field closure is void, and the last such run showed what that does --
# the whole population collapsed into 1 < m < 447 with alpha = -4.7 over 0.9 decades.
# There is no steady cascade there to measure, so there is nothing to compare against.
FRAG = {}
for rule in AGE_RULES4:
    _t = time.time()
    FRAG[rule] = BF.simulate(
        process="fragmentation", system="open", kernel=KERNEL, edges=EDGES,
        ic={"m": M_INJ4, "N": N04}, frag_min_ratio=F_RATIO4,
        frag_split_width=W_SPLIT, frag_age_rule=rule,
        injection_rate=q4, injection_mass=M_INJ4, sink_mass=M_SINK4,
        snapshot_mode="events", snapshot_stride=max(int(MAX_EV4) // 200, 1),
        # Both brakes armed.  stop_sink_events is the physical one and is what the
        # budget was written in; max_events is the wall-clock ceiling.  stop_reason
        # then says which of the two actually bit.
        max_events=MAX_EV4, stop_sink_events=N_OUT4,
        # The gate is NOT 10 here.  In coagulation ten absorptions prove the cascade
        # spans the range; in fragmentation a single deep split can drop a fragment
        # straight onto the sink, and an earlier run had 2099 absorptions while
        # M_out/M_in was 1e-7.  Counted against the budget instead, a fifth of the
        # absorptions is a statement about the steady state rather than about luck.
        iso_start_sink=int(0.2 * N_OUT4),
        # Anchored on the turnover time, not on 1/(K(m_inj,m_inj) N0) -- that is the
        # disruption time of one big body against the other three at t = 0, which has
        # nothing to do with the state being measured.
        iso_age_edges=AGE_EDGES4,
        rng=np.random.default_rng(SEED4), verbose=True)
    r = FRAG[rule]
    print("%-9s %6.1f min | live %d->%d | sink=%d | M_out/M_in=%.3f | stop=%s"
          % (rule, (time.time() - _t) / 60, r["live"][0], r["live"][-1],
             r["sink_events"], r["M_out"][-1] / max(r["M_in"][-1], 1),
             r["meta"]["stop_reason"]))

# The rule is DIAGNOSTIC: it labels particles and nothing else.  Same seed, same
# stream of random numbers, same events -- so these three counters must be equal.
# If they are not, frag_age_rule has leaked into the dynamics and every comparison
# below is between two different simulations rather than two readings of one.
ref = AGE_RULES4[0]
for rule in AGE_RULES4[1:]:
    same = (FRAG[rule]["live"][-1] == FRAG[ref]["live"][-1]
            and FRAG[rule]["sink_events"] == FRAG[ref]["sink_events"]
            and FRAG[rule]["M_sys"][-1] == FRAG[ref]["M_sys"][-1])
    print("dynamics %-9s == %-9s : %s%s"
          % (rule, ref, same, "" if same else "   <== STOP, fix the engine"))

# The age axis is only a measurement once the box has forgotten t = 0.  While this
# fraction is near 100%, every live particle inherits t_born = 0, age equals
# wall-clock time for everyone at once, and the isochrones degrade into "spectrum
# vs time".  It was 2.8% in the last run, which is what made them isochrones.
for rule in AGE_RULES4:
    print("  %-9s %.1f%% of live particles still descend from the IC (t_born = 0)"
          % (rule, 100 * float(np.mean(FRAG[rule]["final_inj_time"] == 0.0))))

KTAG = KERNEL.__name__.replace("kernel_", "")
for rule in AGE_RULES4:
    r = FRAG[rule]
    F4, _ = steady_spectrum(r)
    ir4   = BF.find_inertial_range(r["centers"], F4)          # <== the measurement
    gb4   = BF.guard_band(M_SINK4, M_INJ4, 0.4)
    f4    = BF.fit_powerlaw(r["centers"], F4, *gb4)            # cross-check
    print("%-9s plateau [%9.3g,%9.3g] alpha = %+.3f +- %.3f over %.2f dec | guard %+.3f  R2 = %.3f"
          % (rule, ir4["m_lo"], ir4["m_hi"], ir4["alpha"], ir4["scatter"],
             ir4["decades"], f4["alpha"], f4["r2"]))

    # b is NOT computed here.  The decay law is read in (tau* - tau), and once clocks
    # reset mid-cascade a single age bin mixes tracers whose birth masses differ, so
    # <m>(tau) averaged over the bin is not the trajectory.  Width first (next cell),
    # exponent only after m_reset exists -- see note [11] in the engine.
    add_last_run(r, "open_fragmentation_conservative_%s_f%.2f_%s"
                 % (KTAG, F_RATIO4, rule),
                 analysis=dict(
                     alpha_plateau=ir4["alpha"], alpha_scatter=ir4["scatter"],
                     plateau_decades=ir4["decades"],
                     plateau_m_lo=ir4["m_lo"], plateau_m_hi=ir4["m_hi"],
                     alpha_guard=f4["alpha"], guard_r2=f4["r2"],
                     guard_lo=gb4[0], guard_hi=gb4[1],
                     alpha_theory=PR["alpha"], b_theory=PR["b"],
                     frag_age_rule=rule, f_ratio=F_RATIO4, frag_split_width=W_SPLIT,
                     age_step=AGE_STEP, seed=SEED4, lam=LAM, N0=N04, N_ss=N_SS4,
                     m_inj=M_INJ4, m_sink=M_SINK4,
                     q=r["meta"]["injection_rate"], t_c=t_c4, t_turn=t_turn4,
                     frac_from_ic=float(np.mean(r["final_inj_time"] == 0.0))))

    RESULTS["4 open frag f=%.2g %s" % (F_RATIO4, rule)] = dict(
        b=np.nan, b_th=PR["b"], alpha=ir4["alpha"], alpha_th=PR["alpha"],
        scatter=ir4["scatter"], decades=ir4["decades"], alpha_gb=f4["alpha"])

In [ ]:
N_ISO_SHOW4 = 8      # <<-- isochrones on the spectrum panels; availability is printed


def iso_width(run, min_counts=1e3):
    """(tau, width, <log10 m>, mask): rms spread of log10(m) inside each age bin.

    This is the number the whole sweep is about.  An age class built by 'inherit' is
    a family and is as wide as the inertial range; one built by 'heavier' is a front
    and should not be.  Counts add exactly across mass bins, so this is a property of
    the recorded histogram, not of any fit.
    """
    C  = np.asarray(run["iso_counts"], float)
    lg = np.log10(np.asarray(run["centers"], float))
    S0 = C.sum(axis=1)
    ok = S0 > 0
    m1 = np.full(S0.shape, np.nan); m2 = np.full(S0.shape, np.nan)
    m1[ok] = (C[ok] * lg).sum(axis=1) / S0[ok]
    m2[ok] = (C[ok] * lg**2).sum(axis=1) / S0[ok]
    e = np.asarray(run["iso_age_edges"], float)
    return (np.sqrt(e[:-1] * e[1:]),
            np.sqrt(np.maximum(m2 - m1**2, 0.0)), m1, S0 > min_counts)


COL  = {r: "C%d" % i for i, r in enumerate(AGE_RULES4)}
span = np.log10(M_INJ4 / M_SINK4)          # the inertial range, in decades

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.4))
for rule in AGE_RULES4:
    r = FRAG[rule]
    tau, wid, mlg, k = iso_width(r)
    ax[0].semilogx(tau[k], wid[k], "o-", ms=3, lw=.9, color=COL[rule], label=rule)
    ax[1].loglog(tau[k], 10.0**mlg[k], "o-", ms=3, lw=.9, color=COL[rule], label=rule)
    ages = r["final_t_phys"] - r["final_inj_time"]
    ax[2].hist(np.log10(np.maximum(ages, 1e-30)), bins=60, histtype="step", lw=1.2,
               color=COL[rule], label="%s  med/max = %.3f"
               % (rule, np.median(ages) / max(ages.max(), 1e-30)))
    print("%-9s median width = %.3f dec   (the range spans %.1f dec)   %d age bins used"
          % (rule, np.nanmedian(wid[k]), span, k.sum()))

# The dashed line is the width of a UNIFORM distribution over the whole range: an age
# class sitting on it carries no age information at all.  'inherit' should hug it.
ax[0].axhline(span / np.sqrt(12), ls="--", lw=1, color="k", label="uniform over range")
ax[0].set_xlabel(r"$\tau$"); ax[0].set_ylabel(r"width of $\log_{10}m$  [dec]")
ax[0].legend(fontsize=7); ax[0].set_title("(a) isochrone width -- THE test", fontsize=9)

ax[1].set_xlabel(r"$\tau$"); ax[1].set_ylabel(r"$\langle m\rangle$ of the age class")
ax[1].legend(fontsize=7); ax[1].set_title("(b) where the class sits", fontsize=9)

# med/max = 1 in the legend of (c) is the degeneracy diagnostic: every live particle
# carries the same age, so the "isochrones" are snapshots of the spectrum.
ax[2].set_xlabel(r"$\log_{10}\tau$"); ax[2].set_ylabel("particles")
ax[2].legend(fontsize=6); ax[2].set_title("(c) age distribution", fontsize=9)
fig.tight_layout()


# The spectrum must be IDENTICAL between the rules -- same events, same masses, only
# the labels differ.  Two different alphas here mean the seed is not doing its job.
fig, axs = plt.subplots(1, len(AGE_RULES4), figsize=(5.8 * len(AGE_RULES4), 3.4),
                        squeeze=False)
for i, rule in enumerate(AGE_RULES4):
    r = FRAG[rule]; a = axs[0][i]; c = r["centers"]
    F4, _ = steady_spectrum(r)
    ir4 = BF.find_inertial_range(c, F4)
    idx, tauc, navail = pick_isochrones(r, N_ISO_SHOW4)
    print("%-9s isochrones: %d of %d age bins carry statistics -> drawing %d"
          % (rule, navail, len(tauc), len(idx)))
    draw_isochrones(a, r, idx, tauc)
    # completeness check: the isochrones must add up to the steady state
    iso_sum = r["iso_dndm"].sum(axis=0) / max(r["iso_snapshots"], 1)
    a.loglog(c, np.where(iso_sum > 0, iso_sum * c**2, np.nan), "-", lw=3.5, alpha=.35,
             color="0.4", zorder=2, label="sum of isochrones")
    a.loglog(c, np.where(F4 > 0, F4 * c**2, np.nan), "o", ms=3, color="k", zorder=4,
             label="steady state")
    if np.isfinite(ir4["m_lo"]):
        xs = np.logspace(np.log10(ir4["m_lo"]), np.log10(ir4["m_hi"]), 30)
        a.loglog(xs, anchor_amplitude(c, F4, ir4["m_lo"], ir4["m_hi"], ir4["alpha"])
                 * xs**(ir4["alpha"] + 2), "-", lw=2.2, color="C3",
                 label=r"plateau $\alpha=%.2f$" % ir4["alpha"])
        a.loglog(xs, anchor_amplitude(c, F4, ir4["m_lo"], ir4["m_hi"], PR["alpha"])
                 * xs**(PR["alpha"] + 2), "--", lw=1.4, color="C1",
                 label=r"theory $%.2f$" % PR["alpha"])
    compensated_ylim(a, c, F4)
    a.set_xlim(.3, 3e7); a.set_xlabel("m"); a.set_ylabel(r"$m^2\,dN/dm$")
    a.legend(fontsize=6, loc="lower left")
    a.set_title("%s   f = %.2g, w = %.2g" % (rule, F_RATIO4, W_SPLIT), fontsize=9)
fig.tight_layout()

# Note on alpha from THIS run: at w = 0.05 the spectrum carries a factor-2.3 comb
# every 0.3 dex, and a plateau fitted through it returns a shifted index with a
# small error bar.  Comparing the two rules against each other is fine -- the comb
# is identical in both.  Comparing either against theory is not.

---
## Summary

One row per run. `plateau` is the mean local slope over the longest flat stretch of
$\Gamma(m)$, with its scatter and its width in decades; `guard` is the a-priori band
fit kept as an independent cross-check, and the two are expected to agree. `b` comes
from the isochrones, read as $\langle m\rangle\propto\tau^{\,b}$ for coagulation and
as $\langle m\rangle\propto(\tau_*-\tau)^{\,b}$ for fragmentation.

Read `dec` before believing `plateau`. The expected failure is the fragmentation run
at $f=0$, where the non-local disruption rule drives the drift from the sink scale
and pushes the index towards $-2$ whatever the kernel; its narrow plateau is the
notebook announcing that the number next to it is a diagnosis rather than a
measurement.

Every run above has been written into `runs/`. The analysis notebooks there reload
those files and rebuild each figure separately, so nothing below needs to be re-run
to change a plot.


In [ ]:
hdr = ("%-22s | %7s %7s | %8s %7s %6s | %8s %8s" %
       ("case", "b", "b_th", "plateau", "+-", "dec", "guard", "theory"))
print(hdr); print("-" * len(hdr))
for name, d in RESULTS.items():
    print("%-22s | %7.3f %7.3f | %+8.3f %7.3f %6.2f | %+8.3f %+8.3f" %
          (name, d["b"], d["b_th"], d["alpha"], d["scatter"], d["decades"],
           d["alpha_gb"], d["alpha_th"]))
print("\nplateau = find_inertial_range (longest flat run in the local slope) -- THE answer.")
print("guard   = the a-priori band fit, kept as an independent cross-check.")
print("dec     = plateau width in decades; below ~1 the index is not a measurement.")
print("b = nan  = the age axis was degenerate, see the note in the case-4 cell.")
